# L31 · 后训练总览：SFT 是什么

**学习目标**
- 搞清 AI 模型的三个阶段：预训练 → 后训练(SFT/RL) → 部署
- 理解 SFT（监督微调）：用「指令-回答」数据教模型按格式说话
- 亲手用 numpy 跑一个迷你 SFT，看损失下降

**前置依赖**：L11（梯度下降直觉）、L24（微调概念）  
**预计时长**：55 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行）

---

## 概念讲解：后训练 = 把「通才」训成「好员工」

大模型的生命有三阶段：
1. **预训练**：读全网文本，学会「语言规律」（成本亿级，新手不碰）
2. **后训练（Post-training）**：用精选数据，教它「怎么对话、怎么听话」—— **本课重点**
3. **部署**：上线服务

后训练第一步 **SFT（Supervised Fine-Tuning，监督微调）**：
给模型看大量「用户指令 → 标准回答」范例，让它学会「人类期望的对话格式」。
本课我们用一个极简参数模型，演示 SFT 如何让「输出」逼近目标。

## 第一步：SFT 数据（指令→回答）

In [ ]:
# 极简：把「指令向量」映射到「期望回答向量」
import numpy as np
np.random.seed(0)
X = np.random.randn(50, 4)                 # 50 条指令的特征
W_true = np.array([[1,0,0,0],[0,1,0,0]]).T  # 真实映射(4->2)
Y = X @ W_true + np.random.normal(0, 0.1, (50, 2))   # 期望回答(带噪)
print("数据集：", X.shape[0], "条指令-回答对，输入4维→输出2维")

## 第二步：用梯度下降做 SFT（最小化输出误差）

In [ ]:
W = np.random.randn(4, 2) * 0.1     # 初始随机参数（模型的「大脑」）
lr = 0.05
losses = []
for step in range(200):
    pred = X @ W
    loss = ((pred - Y) ** 2).mean()
    losses.append(loss)
    grad = 2 * X.T @ (pred - Y) / len(X)
    W -= lr * grad
print(f"SFT 后，模型输出误差从 {losses[0]:.3f} 降到 {losses[-1]:.3f}")

# 🎯 AHA 顿悟单元格：看 SFT 把「乱说话」模型训成「会说人话」

运行下面代码。你会看到一张**损失下降曲线**：模型一开始「答非所问」（损失高），
随着 SFT 训练步数增加，它逐渐学会按指令格式输出（损失落到很低）。
这就是 ChatGPT 从「文本续写器」变成「对话助手」的第一步。

> 你刚跑的，正是 SFT 的最小内核。真实 SFT 用的是十亿参数 + 百万指令对，但「用示范数据逼模型学格式」的本质，完全一致。

In [ ]:
# ===== 运行我！看 SFT 损失曲线 =====
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(losses, color="#d62728", lw=2)
plt.title("SFT 训练：模型逐渐学会『按指令回答』（损失下降）")
plt.xlabel("训练步"); plt.ylabel("输出误差（损失）")
plt.grid(alpha=0.3)
plt.show()
print(f"  🎓 SFT 完成：误差 {losses[0]:.3f} → {losses[-1]:.3f}")
print("  ✨ 你亲手完成了一次监督微调 —— 这正是 ChatGPT 训练流水线的第一步！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：须澄清本演示是「线性映射 SFT 的内核投影」，真实 SFT 优化的是 Transformer 权重，目标函数常为交叉熵（next-token）；但「用示范数据最小化输出偏差」本质一致。  
**易错点**：学习率/步数；loss 数值范围随机种子敏感，已固定 seed。  
**AHA 机制**：损失下降曲线，强「模型被训乖了」直观感。  
**衔接**：L32 奖励模型；L33 DPO（更简单的对齐，跳过显式 RM）；L34 PPO；L35 RLHF 串联。  
**依赖**：`pip install numpy matplotlib`。  
**真 LLM 路径**：注明用 `transformers.Trainer` + 指令数据集（如 alpaca）做真 SFT，需 GPU；列为进阶。

# 📚 作业 / 下一步

1. 把 `lr` 改成 0.5，看损失是否震荡（理解学习率敏感性）。
2. 搜索「instruction tuning」「Alpaca 数据集」了解真实 SFT 数据长啥样。
3. 下一课 **L32 奖励模型：教 AI 分辨好坏** —— 训练一个「评委」，给每个回答打分。